In [6]:
from lakehouse import LakeConfig, connect

lake = LakeConfig.from_env()
con = connect()   # DuckDB already authenticated on MinIO (minio:9000)

customers = lake.uri("bronze", "mysql", "customers") + "/*/*.parquet"

df_resultado = con.sql(f"""
    SELECT
        filename,
        COUNT(*) AS quantidade_linhas
    FROM read_parquet('{customers}', filename = true, hive_partitioning = true)
    GROUP BY filename
    ORDER BY filename
""").df()
df_resultado

,filename,quantidade_linhas
0,s3://bronze/mysql/customers/ingestion_date=202...,7906180


In [13]:
# Rows per source table in a unified dataset
con.sql(f"""
    SELECT _source_entity, COUNT(*) AS linhas
    FROM read_parquet('{lake.uri("bronze", "mysql", "emails")}/*/*.parquet', union_by_name = true)
    GROUP BY 1 ORDER BY 1
""").show()

# Columns and types of a dataset
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{customers}')").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬──────────┐
│  _source_entity  │  linhas  │
│     varchar      │  int64   │
├──────────────────┼──────────┤
│ parfois.emails   │       50 │
│ parfois.emails_1 │ 78705879 │
│ parfois.emails_2 │ 78216523 │
│ parfois.emails_3 │ 73418037 │
│ parfois.emails_4 │ 98393713 │
│ parfois.emails_5 │ 86624774 │
│ parfois.emails_6 │ 39430364 │
└──────────────────┴──────────┘

┌─────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name     │ column_type │  null   │   key   │ default │  extra  │
│       varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ CUSTOMER_ID         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ CREATION_DATE       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ SUBSCRIPTION_DATE   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ UNSUBSCRIPTION_DATE │ VARCHAR     │ YES     │ NULL    │ NULL    │ N

In [15]:
# Rows per source table in a unified dataset
con.sql(f"""
    SELECT _source_entity, COUNT(*) AS linhas
    FROM read_parquet('{lake.uri("bronze", "mysql", "sessions")}/*/*.parquet', union_by_name = true)
    GROUP BY 1 ORDER BY 1
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬──────────┐
│   _source_entity   │  linhas  │
│      varchar       │  int64   │
├────────────────────┼──────────┤
│ parfois.sessions_1 │ 31110585 │
│ parfois.sessions_2 │ 44307399 │
│ parfois.sessions_3 │ 47463498 │
│ parfois.sessions_4 │ 58422818 │
└────────────────────┴──────────┘



In [9]:
tabelas = [
    "customers", "emails", "events", "order_cab", 
    "order_det", "paid_media", "product", "sessions"
]

# Gera uma lista de URIs, que o read_parquet do DuckDB aceita diretamente
all_tables_list = [lake.uri("bronze", "mysql", tab) + "/*/*.parquet" for tab in tabelas]

df_resultado = con.sql(f"""
    SELECT
        filename,
        COUNT(*) AS quantidade_linhas
    FROM read_parquet({all_tables_list}, filename = true, hive_partitioning = true)
    GROUP BY filename
    ORDER BY filename
""").df()

df_resultado

,filename,quantidade_linhas
0,s3://bronze/mysql/customers/ingestion_date=202...,7906180
1,s3://bronze/mysql/emails/ingestion_date=2026-1...,50
2,s3://bronze/mysql/emails/ingestion_date=2026-1...,78705879
3,s3://bronze/mysql/emails/ingestion_date=2026-1...,78216523
4,s3://bronze/mysql/emails/ingestion_date=2026-1...,73418037
5,s3://bronze/mysql/emails/ingestion_date=2026-1...,98393713
6,s3://bronze/mysql/emails/ingestion_date=2026-1...,86624774
7,s3://bronze/mysql/emails/ingestion_date=2026-1...,39430364
8,s3://bronze/mysql/events/ingestion_date=2026-1...,810
9,s3://bronze/mysql/order_cab/ingestion_date=202...,9049802
